In [1]:
import pandas as pd

In [2]:
blinkit=pd.read_csv(r"blinkit\blinkit_data.csv")
amazon=pd.read_csv(r"amazon\amazon_data.csv")
zepto=pd.read_csv(r"zepto\zepto_data.csv")

In [3]:
df=pd.concat([blinkit,zepto,amazon],axis=1)
df=df.drop("Unnamed: 0",axis=1)
df.head()

,B_Product_name,B_Price,B_Quantity,Z_Product_name,Z_Price,Z_Quantity,A_Product_name,A_Price,A_Quantity
0,Let's Try Lite Multigrain Mixture Namkeen,₹76,173 g,Nandini Goodlife Toned UHT Milk (Fino Pouch),₹32,1 pack (500 ml),"Amul-Moti Homogenized Toned Milk,\n ...",33,450 Ml
1,Amul Taaza Toned Milk,₹30,500 ml,Nandini Toned Fresh Milk | Pouch,₹24,1 pack (500 ml),Amul-Taaza Homogenised Toned Milk 1\n ...,77,1 L
2,Amul Gold Full Cream Milk,₹72,1 ltr,Nandini Standardized Fresh Milk | Pouch,₹27,1 pack (500 ml),"Amul-Taaza Homogenized Toned Milk,\n ...",38,500 Ml
3,Amul Lactose Free Milk,₹26,250 ml,Nandini Good Life Toned UHT Milk (Fino Pouch),₹15,1 pack (180 ml),Amul-Gold Milk Homogenized\n ...,83,1 Litre
4,Amul Taaza Homogenised Toned Milk,₹77,1 ltr,Amul Gold Full Cream Fresh Milk | Pouch,₹34,1 pack (500 ml),"Mother Dairy-Uht Milk Carton, 1 Liter\n",75,1 Liter


In [4]:
df=df.fillna("N/A")

In [5]:
def check_corr(x):
    if x=="nan": 
        return nan
    l=x.split()[0:5]
    str=" ".join(l[:5])
    return str

In [6]:
l1=df.B_Product_name.apply(check_corr)
l2=df.Z_Product_name.apply(check_corr)
l3=df.A_Product_name.apply(check_corr)

In [7]:
blinkit.iloc[9]

Unnamed: 0                                        9
B_Product_name    Amul Taaza Toned Milk - Pack of 2
B_Price                                         ₹34
B_Quantity                               2 x 200 ml
Name: 9, dtype: object

In [8]:
l1

0            Let's Try Lite Multigrain Mixture
1                        Amul Taaza Toned Milk
2                    Amul Gold Full Cream Milk
3                       Amul Lactose Free Milk
4            Amul Taaza Homogenised Toned Milk
5                     Country Delight Cow Milk
6                     Amul Moti Toned Milk (90
7                      Humpy Farms A2 Cow Milk
8                               Amul Gold Milk
9                      Amul Taaza Toned Milk -
10           Mother Dairy FIT Life Homogenized
11                  Amul Slim 'n' Trim Skimmed
12                   Country Delight 25 g High
13                 Nestle a+ Slim Skimmed Milk
14                     Mother Dairy Toned Milk
15           Mother Dairy FIT Life Homogenized
16                  Amul Buffalo Milk (90 Days
17               Amul Calci+ Calcium Rich High
18                           Mother Dairy Milk
19                   Country Delight 25 g High
20                             Amul Camel Milk
21           

In [9]:
l2

0                 Nandini Goodlife Toned UHT Milk
1                      Nandini Toned Fresh Milk |
2               Nandini Standardized Fresh Milk |
3                     Nandini Good Life Toned UHT
4                      Amul Gold Full Cream Fresh
5               Nandini Standardized Fresh Milk |
6               Amul Taaza Homogenised Toned Milk
7                     Heritage Toned Fresh Milk |
8                Heritage Special Long Life Toned
9                 Nandini Goodlife Toned Milk UHT
10                      Arokya Toned Fresh Milk |
11               Nandini Goodlife Milk UHT (Pouch
12                     Nandini Toned Fresh Milk |
13                    Amul Taaza Toned Fresh Milk
14                  Amul Lactose Free Milk (Tetra
15              Amul Taaza Homogenised Toned Milk
16                 Heritage Full Cream Fresh Milk
17    Akshayakalpa Amrutha A2 Pasteurized Organic
18     Akshayakalpa Organic Pasteurized Cow Fresh
19                 Akshayakalpa Amrutha - A2 Farm


In [10]:
l3

0             Amul-Moti Homogenized Toned Milk, 450
1               Amul-Taaza Homogenised Toned Milk 1
2            Amul-Taaza Homogenized Toned Milk, 500
3        Amul-Gold Milk Homogenized Standardized, 1
4                   Mother Dairy-Uht Milk Carton, 1
5                         Go- Supremo Milk, 1 Litre
6                  Amul-Slim 'N' Trim Skimmed Milk,
7                    Amul-Lactose Free Milk, 250 Ml
8                     Mother Dairy-UHT Fit Lite ESL
9                       Mother Dairy-UHT Milk 180ml
10                -Nestlé LACTOGEN PRO 2, Follow-Up
11                     -Nestlé a+ Toned Milk, Tetra
12              -Nestlé EveryDay Dairy Creamer, 1kg
13                    -Nestlé a+ Slim Skimmed Milk,
14           Aptamil-Gold Infant Formula for Babies
15        -Nestlé Milkmaid Partly Skimmed Sweetened
16       Akshayakalpa Organic-Slim Milk (UHT), Pure
17    Coco Mama-Organic Coconut Milk Dairy-Free,250
18             -Nestlé EveryDay Dairy Creamer, 200g
19          

## Fuzzy name matching with difflib
The `check_corr` approach above only compares the **first 5 words**, so it only catches products whose names *start* identically — that's why so few real matches show up. Below we match on the **full cleaned product name** using `difflib`, which finds ~20 real overlaps instead of ~5.

**Rule applied (as requested):** for each Blinkit (`B`) product, we first look for its best match in Zepto (`Z`). If a good match is found there, we don't also search Amazon for that row — the Amazon (`A`) cell is set to `"N/A"`. Only when no Zepto match clears the cutoff do we look for a match in Amazon. This avoids one Blinkit product getting two separate 'matches' claimed at once.

In [11]:
import re
from difflib import SequenceMatcher, get_close_matches

def clean_name(x):
    """Lowercase, strip HTML artifacts/punctuation, collapse whitespace."""
    if pd.isna(x) or x == "N/A":
        return ""
    x = str(x)
    x = x.replace("\\n", " ").replace("&nbsp;", " ")
    x = re.sub(r"[^A-Za-z0-9\s]", " ", x)   # drop punctuation/commas/pipes
    x = re.sub(r"\s+", " ", x).strip().lower()
    return x

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def best_match(name, choices, cutoff=0.55):
    """Return (matched_choice, score) for the closest string in `choices`,
    or (None, 0.0) if nothing clears the cutoff."""
    if not name:
        return None, 0.0
    hits = get_close_matches(name, choices, n=1, cutoff=cutoff)
    if not hits:
        return None, 0.0
    best = hits[0]
    return best, similarity(name, best)

In [12]:
CUTOFF = 0.55  # tune this up/down until the match count lands around ~20

b_clean = blinkit["B_Product_name"].apply(clean_name)
z_clean = zepto["Z_Product_name"].apply(clean_name)
a_clean = amazon["A_Product_name"].apply(clean_name)

z_choices = z_clean.tolist()
a_choices = a_clean.tolist()

rows = []
for i, b_name in enumerate(b_clean):
    z_match, z_score = best_match(b_name, z_choices, CUTOFF)

    if z_match:
        # Already matched in Zepto -> don't also claim an Amazon match for this row
        z_idx = z_choices.index(z_match)
        z_orig = zepto["Z_Product_name"].iloc[z_idx]
        a_orig, a_score = "N/A", None
    else:
        a_match, a_score = best_match(b_name, a_choices, CUTOFF)
        z_orig = "N/A"
        if a_match:
            a_idx = a_choices.index(a_match)
            a_orig = amazon["A_Product_name"].iloc[a_idx]
        else:
            a_orig, a_score = "N/A", None

    rows.append({
        "B_Product_name": blinkit["B_Product_name"].iloc[i],
        "Z_Product_name": z_orig,
        "Z_score": round(z_score, 2) if z_match else None,
        "A_Product_name": a_orig,
        "A_score": round(a_score, 2) if a_score else None,
    })

matched_df = pd.DataFrame(rows)
matched_df

,B_Product_name,Z_Product_name,Z_score,A_Product_name,A_score
0,Let's Try Lite Multigrain Mixture Namkeen,N/A,NaN,N/A,NaN
1,Amul Taaza Toned Milk,Amul Taaza Toned Fresh Milk | Pouch,0.78,N/A,NaN
2,Amul Gold Full Cream Milk,Amul Gold Full Cream Fresh Milk | Pouch,0.81,N/A,NaN
3,Amul Lactose Free Milk,Amul Lactose Free Milk (Tetra Pack),0.80,N/A,NaN
4,Amul Taaza Homogenised Toned Milk,Amul Taaza Homogenised Toned Milk (Tetra Pack),0.86,N/A,NaN
5,Country Delight Cow Milk,N/A,NaN,N/A,NaN
6,Amul Moti Toned Milk (90 Days Shelf Life),N/A,NaN,"Amul-Moti Homogenized Toned Milk,\n ...",0.60
7,Humpy Farms A2 Cow Milk,N/A,NaN,N/A,NaN
8,Amul Gold Milk,N/A,NaN,N/A,NaN
9,Amul Taaza Toned Milk - Pack of 2,Amul Taaza Toned Fresh Milk | Pouch,0.75,N/A,NaN


In [13]:
# Sanity check: how many real matches did we find in total?
n_matches = ((matched_df["Z_Product_name"] != "N/A") | (matched_df["A_Product_name"] != "N/A")).sum()
print(f"Total matched rows: {n_matches} out of {len(matched_df)}")

Total matched rows: 12 out of 36


In [14]:
matched_df

,B_Product_name,Z_Product_name,Z_score,A_Product_name,A_score
0,Let's Try Lite Multigrain Mixture Namkeen,N/A,NaN,N/A,NaN
1,Amul Taaza Toned Milk,Amul Taaza Toned Fresh Milk | Pouch,0.78,N/A,NaN
2,Amul Gold Full Cream Milk,Amul Gold Full Cream Fresh Milk | Pouch,0.81,N/A,NaN
3,Amul Lactose Free Milk,Amul Lactose Free Milk (Tetra Pack),0.80,N/A,NaN
4,Amul Taaza Homogenised Toned Milk,Amul Taaza Homogenised Toned Milk (Tetra Pack),0.86,N/A,NaN
5,Country Delight Cow Milk,N/A,NaN,N/A,NaN
6,Amul Moti Toned Milk (90 Days Shelf Life),N/A,NaN,"Amul-Moti Homogenized Toned Milk,\n ...",0.60
7,Humpy Farms A2 Cow Milk,N/A,NaN,N/A,NaN
8,Amul Gold Milk,N/A,NaN,N/A,NaN
9,Amul Taaza Toned Milk - Pack of 2,Amul Taaza Toned Fresh Milk | Pouch,0.75,N/A,NaN
